In [1]:
begin
    using Pkg
    dev_folder = @__DIR__
    Pkg.activate(dev_folder)
end

# ONLY run this block once to set up the development environment
# begin
#     pkg_folder = joinpath(dev_folder, "..")
#     Pkg.develop(path=pkg_folder)
#     Pkg.instantiate()
# end

Threads.nthreads()

  Activating project at `~/Realizibility_index/BindingAndCatalysis.jl/Examples`


24

In [3]:
# using Revise
using BindingAndCatalysis # import the package
using CairoMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...

In [4]:
model = let
    N = [1 1 0 0 0 -1 0 0 0 0 0 0 0 0 0;
         1 0 1 0 0 0 -1 0 0 0 0 0 0 0 0;
         1 0 0 1 0 0 0 -1 0 0 0 0 0 0 0;
         1 0 0 0 1 0 0 0 -1 0 0 0 0 0 0;
         0 1 1 0 0 0 0 0 0 -1 0 0 0 0 0;
         0 1 0 1 0 0 0 0 0 0 -1 0 0 0 0;
         0 1 0 0 1 0 0 0 0 0 0 -1 0 0 0;
         0 0 1 1 0 0 0 0 0 0 0 0 -1 0 0;
         0 0 1 0 1 0 0 0 0 0 0 0 0 -1 0;
         0 0 0 1 1 0 0 0 0 0 0 0 0 0 -1;] # define stoichiometry matrix
    x_sym = [:A, :B, :C, :D, :E, :ab, :ac, :ad, :ae, :bc, :bd, :be, :cd, :ce, :de] # Optional: define species symbols
    q_sym = [:tA, :tB, :tC, :tD, :tE] # Optional: define total concentration symbols
    K_sym = [:K12, :K13, :K14, :K15, :K23, :K24, :K25, :K34, :K35, :K45] # Optional: define binding constant symbols
    Bnc(N = N, x_sym=x_sym, q_sym=q_sym, K_sym=K_sym) # create Bnc model
end

----------Binding Network Summary:-------------
Number of species (n): 15
Number of conserved quantities (d): 5
Number of reactions (r): 10
L matrix: sparse([1, 2, 3, 4, 5, 1, 2, 1, 3, 1  …  2, 4, 2, 5, 3, 4, 3, 5, 4, 5], [1, 2, 3, 4, 5, 6, 6, 7, 7, 8  …  11, 11, 12, 12, 13, 13, 14, 14, 15, 15], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 5, 15)
N matrix: sparse([1, 2, 3, 4, 1, 5, 6, 7, 2, 5  …  1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  6, 7, 8, 9, 10, 11, 12, 13, 14, 15], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 10, 15)
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [5]:
find_all_vertices!(model;mode=:float) # find all possible vertices

[ Info: ---------------------Start finding all regimes--------------------
[ Info: Finished, with 2451 regimes found and 2451 asymptotic regimes.
[ Info: 2.Building x-neighbor regime graph...
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
[ Info: 3.Building regime objects...
[ Info: 4.Propagating affine data and deferred nullity labels...
[ Info: Calculating vertices neighbor graph with qK change dir
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
[ Info: Finished.


In [6]:
summary(model) # Now regime data is available

----------Binding Network Summary:-------------
Number of species (n): 15
Number of conserved quantities (d): 5
Number of reactions (r): 10
L matrix: sparse([1, 2, 3, 4, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4, 2, 5, 3, 4, 3, 5, 4, 5], [1, 2, 3, 4, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 5, 15)
N matrix: sparse([1, 2, 3, 4, 1, 5, 6, 7, 2, 5, 8, 9, 3, 6, 8, 10, 4, 7, 9, 10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 10, 15)
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: Yes
Number of regimes: 2451
  - Invertible + Asymptotic: 1296
  - Singular +  Asymptotic: 1155
  - Invertible +  Non-Asymptotic: 0
  - Singular +  Non-Asymptotic: 0
------------------------

In [7]:
vtx_grh = get_vertices_graph!(model, full=true)

VertexGraph with 2451 vertices and 42900 edges

In [8]:
siso = get_SISO_graph(model, :tA) # will give you a simple DiGraph

{2451, 7488} directed simple Int64 graph

In [9]:
pths = SISOPaths(model, :tA) # get all SISO paths for tS

[ Info: sources: [1144, 2261, 1703, 1028, 1438, 1812, 1437, 525, 2232, 1704, 694, 1050, 2188, 844, 358, 1058, 1814, 1102, 79, 1905, 2030, 2074, 1570, 1117, 528, 535, 1896, 839, 250, 2086, 1526, 1939, 1052, 2413, 1245, 828, 1138, 1848, 1155, 1111, 595, 1251, 325, 1621, 444, 2044, 351, 1808, 1236, 1070, 2279, 1580, 1004, 1255, 1336, 18, 1836, 2439, 2241, 987, 1578, 2000, 61, 841, 876, 900, 1585, 1719, 1528, 1325, 1019, 1354, 1519, 80, 51, 2058, 667, 2190, 2029, 2418, 1120, 1908, 2281, 2182, 650, 1395, 2406, 90, 1811, 1941, 2357, 1116, 658, 599, 1714, 487, 406, 643, 2263, 836, 261, 2301, 2043, 1506, 1582, 340, 1535, 1872, 1246, 1152, 1409, 1560, 1996, 645, 649, 1141, 988, 1009, 2090, 14, 1854, 1538, 2257, 2313, 1723, 592, 2265, 597, 879, 1063, 1235, 408, 324, 1151, 698, 2371, 54, 1479, 483, 1576, 961, 903, 1256, 1993, 1998, 1909, 1937, 693, 2042, 1384, 1562, 632, 902, 1607, 373, 53, 2081, 2062, 2233, 1529, 1797, 594, 642, 1041, 1813, 826, 1595, 1442, 1123, 665, 940, 479, 1850, 2076, 1434,

SISOPaths object with 6243216 paths for qK coordinate index 1

In [12]:
# enumerate all SISO paths for increasing tS, note if set show_volume=true, 
# a lot of time will be used to compute the condtion polyhedron and volumes 
summary(pths; show_volume=false)

Path 1         #1 → #2 → #7 → #32 → #155                                                                                                                                 
Path 2         #1 → #2 → #7 → #130 → #155                                                                                                                                
Path 3         #1 → #2 → #27 → #32 → #155                                                                                                                                
Path 4         #1 → #2 → #27 → #150 → #155                                                                                                                               
Path 5         #1 → #2 → #125 → #130 → #155                                                                                                                              
Path 6         #1 → #2 → #125 → #150 → #155                                                                                                           

Excessive output truncated after 524356 bytes.

Path 2819      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #207 → #132 → #130 → #155                                                                          
Path 2820      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #207 → #132 → #157 → #154 → #155                                                                   
Path 2821      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #207 → #132 → #157 → #155                                                                          
Path 2822      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #207 → #157 → #154 → #155                                                                          
Path 2823      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #207 → #157 → #155                                                                                 
Path 2824      #13 → #136 → #137 → #212 → #209 → #213 → #203 → #208 → #232 → #228 → #154 → #155                                                       

LoadError: InterruptException:

In [ ]:
summary_RO_path(pths;observe_x=5, deduplicate=true, keep_nonasymptotic=false,keep_singular=false)